In [1]:
import pandas, geopandas, transformers, torch
print("All packages loaded successfully!")


c:\Users\patrizia\Desktop\GSNA_SEM_g1\v-PythonEnvironment\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


All packages loaded successfully!


In [8]:
import pandas as pd
import geopandas as gpd
from transformers import pipeline
import torch
import os

# Paths
INPUT_PARQUET = r"C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\data_GSNA_PRJ_ST26_G1_20260519\Greece.Wildfires2021.parquet"
UNCLASSIFIED_GEOJSON = r"C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\UNCLASSIFIED_GEOJSON\Greece.Wildfires2021_unclassified.geojson"
CLASSIFIED_GEOJSON = r"C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\CLASSIFIED_GEOJSON\Greece.Wildfires2021_classified.geojson"
CLASSIFIED_GEOPARQUET = r"C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\CLASSIFIED_GEOPARQUET\Greece.Wildfires2021_classified.geoparquet"
MODEL_PATH = r"C:\Users\patrizia\Desktop\GSNA_SEM_g1\P2_SemanticClassification\disaster-twitter-xlm-ROBERTA-al"

print("✓ Paths configured successfully")

✓ Paths configured successfully


In [9]:
def preprocess(text: str) -> str:
    """Pre-process texts by replacing usernames and links with placeholders.
    This is required for the disaster-twitter model as specified in the README.
    """
    new_text: list[str] = []
    for t in text.split(" "):
        t: str = '@user' if t.startswith('@') and len(t) > 1 else t
        t = 'http' if t.startswith('http') else t
        new_text.append(t)
    return " ".join(new_text)

print("✓ Preprocessing function defined")

✓ Preprocessing function defined


In [10]:
print(f"Loading data from {INPUT_PARQUET}...")
df = pd.read_parquet(INPUT_PARQUET)

print(f"Total records loaded: {len(df)}")
print(f"Columns: {df.columns.tolist()}")

# Check if geometry column exists
if 'geom' in df.columns:
    print("✓ Geometry column found")

    # Convert WKB -> GeoSeries
    df['geom'] = gpd.GeoSeries.from_wkb(df['geom'])

    # Create GeoDataFrame using existing geometry
    print(f"\nCreating GeoDataFrame...")
    gdf = gpd.GeoDataFrame(
        df,
        geometry='geom',
        crs="EPSG:4326"
    )

    print(f"✓ GeoDataFrame created with {len(gdf)} records")

else:
    raise ValueError("No 'geom' column found in parquet file")

Loading data from C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\data_GSNA_PRJ_ST26_G1_20260519\Greece.Wildfires2021.parquet...
Total records loaded: 371
Columns: ['message_id', 'date', 'text', 'tags', 'tweet_lang', 'source', 'place', 'geom', 'retweets', 'tweet_favorites', 'photo_url', 'quoted_status_id', 'user_id', 'user_name', 'user_location', 'followers', 'friends', 'user_favorites', 'status', 'user_lang', 'latitude', 'longitude', 'data_source']
✓ Geometry column found

Creating GeoDataFrame...
✓ GeoDataFrame created with 371 records


In [12]:
# Convert datetime and complex columns to string for GeoJSON compatibility
print("Converting columns for GeoJSON export...")
for col in gdf.columns:
    if pd.api.types.is_datetime64_any_dtype(gdf[col]):
        gdf[col] = gdf[col].dt.strftime('%Y-%m-%dT%H:%M:%S')
    elif gdf[col].apply(lambda x: isinstance(x, (list, dict))).any():
        gdf[col] = gdf[col].astype(str)

print(f"Exporting unclassified data to {UNCLASSIFIED_GEOJSON}...")
try:
    gdf.to_file(UNCLASSIFIED_GEOJSON, driver="GeoJSON")
    print(f"✓ Unclassified GeoJSON exported successfully")
except Exception as e:
    print(f"✗ Error saving unclassified GeoJSON: {e}")

Converting columns for GeoJSON export...
Exporting unclassified data to C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\UNCLASSIFIED_GEOJSON\Greece.Wildfires2021_unclassified.geojson...
✓ Unclassified GeoJSON exported successfully


In [13]:
print(f"Loading RoBERTa model from {MODEL_PATH}...")
device = 0 if torch.cuda.is_available() else -1
print(f"Using device: {'GPU' if device == 0 else 'CPU'}")

# Use the correct tokenizer as specified in README
classifier = pipeline(
    "text-classification", 
    model=MODEL_PATH, 
    tokenizer='cardiffnlp/twitter-xlm-roberta-base',  # Correct tokenizer
    device=device, 
    truncation=True, 
    max_length=512
)
print("✓ RoBERTa model loaded successfully")

print("\nClassifying tweets...")
texts = gdf['text'].fillna("").tolist()
print(f"Total tweets to classify: {len(texts)}")

if not texts:
    print("⚠ No texts to classify!")
    results = []
else:
    # Apply preprocessing as required by the model
    print("Preprocessing texts (replacing @users and URLs)...")
    preprocessed_texts = [preprocess(text) for text in texts]
    
    results = classifier(preprocessed_texts, batch_size=32)
    print(f"✓ Classification completed with preprocessing")

# Extract labels and scores
labels = [res['label'] for res in results]
scores = [res['score'] for res in results]

gdf['classification_label'] = labels
gdf['classification_score'] = scores

# Map to binary status
gdf['classification_status'] = gdf['classification_label'].apply(
    lambda x: 1 if x in ['LABEL_1', '1', 'disaster', 'related'] else 0
)

print(f"\nClassification results:")
print(f"  Disaster-related (1): {(gdf['classification_status'] == 1).sum()}")
print(f"  Not disaster-related (0): {(gdf['classification_status'] == 0).sum()}")

Loading RoBERTa model from C:\Users\patrizia\Desktop\GSNA_SEM_g1\P2_SemanticClassification\disaster-twitter-xlm-ROBERTA-al...
Using device: CPU


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 4749.26it/s]


✓ RoBERTa model loaded successfully

Classifying tweets...
Total tweets to classify: 371
Preprocessing texts (replacing @users and URLs)...
✓ Classification completed with preprocessing

Classification results:
  Disaster-related (1): 27
  Not disaster-related (0): 344


In [14]:
print(f"\nExporting classified data to {CLASSIFIED_GEOJSON}...")
try:
    gdf.to_file(CLASSIFIED_GEOJSON, driver="GeoJSON")
    print(f"✓ Classified GeoJSON exported successfully")
except Exception as e:
    print(f"✗ Error saving classified GeoJSON: {e}")

print(f"Exporting classified data to {CLASSIFIED_GEOPARQUET}...")
try:
    gdf.to_parquet(CLASSIFIED_GEOPARQUET)
    print(f"✓ Classified GeoParquet exported successfully")
except Exception as e:
    print(f"✗ Error saving classified GeoParquet: {e}")

print("\n" + "="*50)
print("✓ PROCESSING COMPLETE!")
print("="*50)
print(f"\nResults saved to:")
print(f"  • {CLASSIFIED_GEOJSON}")
print(f"  • {CLASSIFIED_GEOPARQUET}")
print(f"\nTotal classified records: {len(gdf)}")


Exporting classified data to C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\CLASSIFIED_GEOJSON\Greece.Wildfires2021_classified.geojson...
✓ Classified GeoJSON exported successfully
Exporting classified data to C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\CLASSIFIED_GEOPARQUET\Greece.Wildfires2021_classified.geoparquet...
✓ Classified GeoParquet exported successfully

✓ PROCESSING COMPLETE!

Results saved to:
  • C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\CLASSIFIED_GEOJSON\Greece.Wildfires2021_classified.geojson
  • C:\Users\patrizia\Desktop\GSNA_SEM_g1\P1_DataAcquisition-and-Pre-Filtering\CLASSIFIED_GEOPARQUET\Greece.Wildfires2021_classified.geoparquet

Total classified records: 371
